# Matter (connectedhomeip) GitHub issues: download and build one CSV

This notebook downloads **real data** from the public GitHub repository
[project-chip/connectedhomeip](https://github.com/project-chip/connectedhomeip) and builds:

* `data/processed/matter_issues_full.csv`: **one file, one row per issue**, with the issue text, all its
  comments, labels, dates, the pull request that fixed it, the outcome, and duplicate links
* `data/processed/overview.pdf`: a short overview for the professor
* `data/raw/download_<date>/`: every raw API response as JSON, kept as the untouched original

**How to run:** run the cells from top to bottom (Shift+Enter). Each step is **resumable**. If a cell stops
(network error, laptop sleep), run the same cell again and it continues where it stopped.
Data that is already downloaded is never downloaded again.

**Token:** the GitHub token is read from the `GITHUB_TOKEN` environment variable. If it is not set, the notebook asks for
it in a hidden input box. It is kept only in memory, never printed, and never saved.

No models are trained in this notebook.

## 1. Setup

In [1]:
import os, json, time, re, datetime, textwrap, urllib.parse
from collections import defaultdict
from getpass import getpass
from pathlib import Path

import requests
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

OWNER, REPO = "project-chip", "connectedhomeip"
REPO_FULL = f"{OWNER}/{REPO}"
API = "https://api.github.com"
GRAPHQL_URL = "https://api.github.com/graphql"

DATA = Path(os.environ.get("MATTER_DATA_DIR", "data"))   # all output goes here
RAW = DATA / "raw"
PROCESSED = DATA / "processed"
RUN_FILE = RAW / "current_run.json"
OUTPUT_CSV = PROCESSED / "matter_issues_full.csv"

RATE_LIMIT_FLOOR = 50   # pause when fewer API requests than this are left
GRAPHQL_BATCH = 50      # closed issues per GraphQL query

In [2]:
def log(msg):
    print(f"[{datetime.datetime.now():%H:%M:%S}] {msg}", flush=True)


def write_json(path, obj):
    """Write atomically, so a crash never leaves a half-written file."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(obj, ensure_ascii=False))
    tmp.replace(path)


def read_json(path):
    return json.loads(Path(path).read_text())


def load_pages(folder):
    """All page_XXXXX.json files of a folder, joined in page order."""
    items = []
    for p in sorted(Path(folder).glob("page_*.json")):
        items.extend(read_json(p))
    return items


def sleep_until_reset(resp, reason):
    reset = resp.headers.get("x-ratelimit-reset")
    wait = max(0, int(reset) - time.time()) + 5 if reset else 60
    log(f"{reason}: sleeping {wait:.0f}s until the rate limit resets ...")
    time.sleep(wait)


def request(method, url, max_retries=6, **kwargs):
    """GitHub request with retries on errors and automatic waiting for the rate limit."""
    for attempt in range(1, max_retries + 1):
        try:
            resp = session.request(method, url, timeout=90, **kwargs)
        except requests.RequestException as e:
            log(f"Network error ({type(e).__name__}), retry {attempt}/{max_retries} in {5 * attempt}s")
            time.sleep(5 * attempt)
            continue
        remaining = resp.headers.get("x-ratelimit-remaining")
        if resp.status_code in (403, 429):
            if "retry-after" in resp.headers:            # secondary rate limit
                time.sleep(int(resp.headers["retry-after"]) + 1)
                continue
            if remaining == "0":
                sleep_until_reset(resp, "Rate limit exhausted")
                continue
            if "rate limit" in resp.text.lower():
                time.sleep(60 * attempt)
                continue
        if resp.status_code >= 500 or resp.status_code == 408:
            log(f"HTTP {resp.status_code}, retry {attempt}/{max_retries} in {5 * attempt}s")
            time.sleep(5 * attempt)
            continue
        if remaining is not None and int(remaining) < RATE_LIMIT_FLOOR:
            sleep_until_reset(resp, f"Only {remaining} requests left")
        return resp
    raise RuntimeError(f"Giving up on {method} {url} after {max_retries} attempts")

## 2. Connect to GitHub
Checks the token and shows how many API requests are left (5,000 per hour).

In [3]:
_token = os.environ.get("GITHUB_TOKEN") or getpass("GitHub token (hidden, not saved): ")
session = requests.Session()
session.headers.update({
    "Authorization": f"Bearer {_token}",
    "Accept": "application/vnd.github+json",
    "X-GitHub-Api-Version": "2022-11-28",
    "User-Agent": "matter-issues-research",
})
del _token

r = request("GET", f"{API}/rate_limit")
r.raise_for_status()
limits = r.json()["resources"]
print("Token OK. Requests left - REST:", limits["core"]["remaining"], "| GraphQL:", limits["graphql"]["remaining"])

Token OK. Requests left - REST: 4261 | GraphQL: 5000


## 3. Choose the download folder
An unfinished download is reused, so a restart resumes it even on a later day.
Set `FRESH = True` to start a completely new download in a folder with today's date.

In [4]:
FRESH = False

if RUN_FILE.exists() and not FRESH and (RAW / read_json(RUN_FILE)["folder"]).exists():
    run_dir = RAW / read_json(RUN_FILE)["folder"]
else:
    run_dir = RAW / f"download_{datetime.date.today().isoformat()}"
    run_dir.mkdir(parents=True, exist_ok=True)
    write_json(RUN_FILE, {"folder": run_dir.name, "started": datetime.datetime.now().isoformat(timespec="seconds")})

download_date = read_json(RUN_FILE)["started"][:10]
print("Raw data folder:", run_dir, "| download started:", download_date)

Raw data folder: data/raw/download_2026-11-11 | download started: 2026-11-11


In [5]:
def download_paginated(out_dir, url, params):
    """Follow GitHub's 'next' links page by page; every page is saved. Progress is in _state.json."""
    out_dir.mkdir(parents=True, exist_ok=True)
    state_file = out_dir / "_state.json"
    if state_file.exists():
        state = read_json(state_file)
        if state["done"]:
            log(f"{out_dir.name}: already complete ({state['items']:,} items in {state['page']} pages)")
            return
        log(f"{out_dir.name}: resuming after page {state['page']}")
    else:
        state = {"page": 0, "items": 0, "next_url": f"{url}?{urllib.parse.urlencode(params)}", "done": False}

    while state["next_url"]:
        resp = request("GET", state["next_url"])
        if resp.status_code != 200:
            raise RuntimeError(f"HTTP {resp.status_code}: {resp.text[:300]}")
        data = resp.json()
        state["page"] += 1
        state["items"] += len(data)
        write_json(out_dir / f"page_{state['page']:05d}.json", data)
        state["next_url"] = resp.links.get("next", {}).get("url")
        state["done"] = state["next_url"] is None
        write_json(state_file, state)
        if state["page"] % 10 == 0 or state["done"]:
            log(f"{out_dir.name}: page {state['page']} ({state['items']:,} items, "
                f"{resp.headers.get('x-ratelimit-remaining', '?')} requests left)")
    log(f"{out_dir.name}: done")

## 4. Download all issues (and pull requests)
`GET /repos/project-chip/connectedhomeip/issues?state=all`. This endpoint returns issues **and** pull requests.
Items with a `pull_request` field are pull requests. Takes about 20 minutes.

In [6]:
download_paginated(run_dir / "issues", f"{API}/repos/{OWNER}/{REPO}/issues",
                   {"state": "all", "per_page": 100, "sort": "created", "direction": "asc"})

[23:16:51] issues: already complete (47,539 items in 476 pages)


In [7]:
# Split into issues and pull requests (deduplicated by number) and keep a copy of each.
_items = {i["number"]: i for i in load_pages(run_dir / "issues")}.values()
issues_raw = sorted((i for i in _items if "pull_request" not in i), key=lambda i: i["created_at"])
prs_raw = sorted((i for i in _items if "pull_request" in i), key=lambda i: i["created_at"])
write_json(run_dir / "split" / "issues_only.json", issues_raw)
write_json(run_dir / "split" / "pull_requests.json", prs_raw)
print(f"{len(issues_raw):,} issues, {len(prs_raw):,} pull requests")

14,698 issues, 32,841 pull requests


## 5. Download all comments
`GET /repos/project-chip/connectedhomeip/issues/comments` returns comments for the whole repository
(issues and pull requests).

**Important:** GitHub returns at most 300 pages (30,000 comments) for one listing, and this repository has many more.
So the comments are sorted by `updated_at`. Each time a listing reaches that limit, a new listing starts from the last
`updated_at` seen (`since=...`). The few comments that appear twice at the overlap are removed later by their `id`.
This is the slowest step: about 1-2 hours.

In [8]:
def download_all_comments():
    out_dir = run_dir / "comments"
    out_dir.mkdir(parents=True, exist_ok=True)
    state_file = out_dir / "_state.json"

    def listing_url(since):
        params = {"per_page": 100, "sort": "updated", "direction": "asc"}
        if since:
            params["since"] = since
        return f"{API}/repos/{OWNER}/{REPO}/issues/comments?{urllib.parse.urlencode(params)}"

    if state_file.exists():
        state = read_json(state_file)
        if state["done"]:
            log(f"comments: already complete ({state['items']:,} rows in {state['page']} pages)")
            return
        log(f"comments: resuming after page {state['page']}")
    else:
        state = {"page": 0, "items": 0, "listing": 1, "since": None, "next_url": listing_url(None), "done": False}

    while not state["done"]:
        resp = request("GET", state["next_url"])
        if resp.status_code != 200:
            raise RuntimeError(f"HTTP {resp.status_code}: {resp.text[:300]}")
        data = resp.json()
        state["page"] += 1
        state["items"] += len(data)
        write_json(out_dir / f"page_{state['page']:05d}.json", data)
        next_url = resp.links.get("next", {}).get("url")
        if next_url:
            state["next_url"] = next_url
        elif len(data) == 100:
            # A full last page means GitHub's 30,000 cap stopped the listing, not the end of the data.
            since = max(c["updated_at"] for c in data)
            if since == state["since"]:   # 100+ comments with the same timestamp: step past it
                since = (pd.Timestamp(since) + pd.Timedelta(seconds=1)).strftime("%Y-%m-%dT%H:%M:%SZ")
            state.update(since=since, listing=state["listing"] + 1, next_url=listing_url(since))
            log(f"comments: listing limit reached, continuing from updated_at >= {since} (listing {state['listing']})")
        else:
            state.update(next_url=None, done=True)
        write_json(state_file, state)
        if state["page"] % 10 == 0 or state["done"]:
            log(f"comments: page {state['page']} ({state['items']:,} rows, up to {data[-1]['updated_at'][:10] if data else '-'}, "
                f"{resp.headers.get('x-ratelimit-remaining', '?')} requests left)")
    log("comments: done")


download_all_comments()

[23:16:54] comments: already complete (142,949 rows in 1430 pages)


## 6. Which pull request fixed which issue
For every closed issue, GraphQL `closedByPullRequestsReferences` lists the PRs linked as closing it,
with 50 issues per query. Issues that GraphQL cannot answer are fetched from the
`/issues/{number}/timeline` endpoint instead (fallback).

In [9]:
def graphql_query(numbers):
    fields = "\n".join(
        f"i{n}: issue(number: {n}) {{ number closedByPullRequestsReferences(first: 25, includeClosedPrs: true) "
        f"{{ totalCount nodes {{ number url state merged mergedAt repository {{ nameWithOwner }} }} }} }}"
        for n in numbers)
    return (f'query {{ rateLimit {{ cost remaining resetAt }} '
            f'repository(owner: "{OWNER}", name: "{REPO}") {{ {fields} }} }}')


closed_numbers = sorted(i["number"] for i in issues_raw if i["state"] == "closed")
batches = [closed_numbers[k:k + GRAPHQL_BATCH] for k in range(0, len(closed_numbers), GRAPHQL_BATCH)]
gql_dir = run_dir / "graphql"
gql_dir.mkdir(parents=True, exist_ok=True)
fallback_file = run_dir / "timeline" / "_needs_timeline.json"
needs_timeline = set(read_json(fallback_file)) if fallback_file.exists() else set()

log(f"{len(closed_numbers):,} closed issues in {len(batches)} batches")
for b, numbers in enumerate(batches, 1):
    path = gql_dir / f"batch_{numbers[0]:06d}_{numbers[-1]:06d}.json"
    if path.exists():
        continue                                   # already downloaded
    try:
        resp = request("POST", GRAPHQL_URL, json={"query": graphql_query(numbers)})
        body = resp.json() if resp.headers.get("content-type", "").startswith("application/json") else None
    except RuntimeError as e:
        log(f"batch {b} failed: {e}")
        resp, body = None, None
    if resp is None or resp.status_code != 200 or body is None or body.get("data") is None:
        log(f"batch {b}/{len(batches)} failed; these {len(numbers)} issues will use the timeline fallback")
        needs_timeline.update(numbers)
        if body is not None:
            write_json(gql_dir / f"failed_{numbers[0]:06d}_{numbers[-1]:06d}.json", body)
    else:
        write_json(path, body)
        repo_data = body["data"].get("repository") or {}
        needs_timeline.update(n for n in numbers if not repo_data.get(f"i{n}"))
        if b % 10 == 0 or b == len(batches):
            log(f"batch {b}/{len(batches)} (GraphQL points left: {(body['data'].get('rateLimit') or {}).get('remaining', '?')})")
    write_json(fallback_file, sorted(needs_timeline))
log("GraphQL done")

[23:16:54] 12,205 closed issues in 245 batches


[23:16:54] GraphQL done


In [10]:
# Timeline fallback, only for issues GraphQL could not answer (often none).
log(f"Timeline fallback for {len(needs_timeline)} issues")
for k, n in enumerate(sorted(needs_timeline), 1):
    path = run_dir / "timeline" / f"issue_{n}.json"
    if path.exists():
        continue
    events, url = [], f"{API}/repos/{OWNER}/{REPO}/issues/{n}/timeline?per_page=100"
    while url:
        resp = request("GET", url)
        if resp.status_code != 200:
            log(f"timeline #{n}: HTTP {resp.status_code}, skipped")
            events = None
            break
        events.extend(resp.json())
        url = resp.links.get("next", {}).get("url")
    if events is not None:
        write_json(path, events)
    if k % 25 == 0:
        log(f"timeline {k}/{len(needs_timeline)}")
log("Download complete.")

[23:16:54] Timeline fallback for 0 issues


[23:16:54] Download complete.


## 7. Build the table
From here on nothing is downloaded. The cells only read the raw files saved above, so you can re-run them at any time.

### 7a. Load the raw data

In [11]:
_items = {i["number"]: i for i in load_pages(run_dir / "issues")}.values()
issues_raw = sorted((i for i in _items if "pull_request" not in i), key=lambda i: i["created_at"])
prs_raw = sorted((i for i in _items if "pull_request" in i), key=lambda i: i["created_at"])
comments_raw = sorted({c["id"]: c for c in load_pages(run_dir / "comments")}.values(), key=lambda c: c["created_at"])
issue_numbers = {i["number"] for i in issues_raw}
print(f"{len(issues_raw):,} issues | {len(prs_raw):,} pull requests | {len(comments_raw):,} comments (whole repo)")

14,698 issues | 32,841 pull requests | 142,945 comments (whole repo)


### 7b. Anonymise usernames
Every login becomes an ID such as `user_00042` (`bot_00007` for bot accounts), numbered in order of first appearance.
`@mentions` of these logins inside text are replaced too. The raw JSON files keep the real names, so do not share them.

In [12]:
MENTION_RE = re.compile(r"(?<![\w/.`-])@([A-Za-z0-9](?:[A-Za-z0-9-]{0,38}))")
user_ids = {}


def is_bot(user):
    return bool(user) and (user.get("type") == "Bot" or user.get("login", "").endswith("[bot]"))


def add_user(user):
    if user and user.get("login") and user["login"].lower() not in user_ids:
        user_ids[user["login"].lower()] = f"{'bot' if is_bot(user) else 'user'}_{len(user_ids) + 1:05d}"


def anon_user(user):
    return user_ids.get(user["login"].lower()) if user and user.get("login") else None


def anon_text(s):
    if not s:
        return ""
    return MENTION_RE.sub(lambda m: "@" + user_ids[m.group(1).lower()] if m.group(1).lower() in user_ids else m.group(0), s)


for i in issues_raw:
    add_user(i["user"])
for c in comments_raw:
    add_user(c["user"])
for i in issues_raw + prs_raw:
    add_user(i["user"])
    for a in i.get("assignees") or []:
        add_user(a)
print(f"{len(user_ids):,} distinct accounts anonymised")

2,161 distinct accounts anonymised


### 7c. Comments per issue (comments on pull requests are left out)

In [13]:
comments_by_issue = defaultdict(list)
for c in comments_raw:
    n = int(c["issue_url"].rstrip("/").rsplit("/", 1)[1])
    if n in issue_numbers:
        comments_by_issue[n].append({
            "id": c["id"], "author": anon_user(c["user"]), "bot": is_bot(c["user"]),
            "created_at": c["created_at"], "body": anon_text(c["body"]),
        })
n_issue_comments = sum(len(v) for v in comments_by_issue.values())
print(f"{n_issue_comments:,} comments belong to issues")

30,379 comments belong to issues


### 7d. Fix links (GraphQL results first, timeline fallback for the rest)

In [14]:
ISSUE_URL = rf"https?://github\.com/{OWNER}/{REPO}/(?:issues|pull)/"
fix_links, link_source = {}, {}

# Fallback: a PR that cross-references the issue with a closing keyword ("fixes #N", "closes #N", ...).
for p in sorted((run_dir / "timeline").glob("issue_*.json")):
    n = int(p.stem.split("_")[1])
    kw = r"\b(?:close[sd]?|fix(?:e[sd])?|resolve[sd]?)\s*:?\s+"
    pat = re.compile(rf"{kw}(?:{re.escape(REPO_FULL)})?#{n}\b|{kw}{ISSUE_URL}{n}\b", re.IGNORECASE)
    found = {}
    for ev in read_json(p):
        src = (ev.get("source") or {}).get("issue") or {}
        if ev.get("event") == "cross-referenced" and "pull_request" in src \
                and pat.search(f"{src.get('title') or ''}\n{src.get('body') or ''}"):
            repo = (src.get("repository") or {}).get("full_name", REPO_FULL)
            found[(repo, src["number"])] = {"pr": src["number"], "repo": repo,
                                            "merged": bool(src["pull_request"].get("merged_at"))}
    fix_links[n], link_source[n] = list(found.values()), "timeline"

for p in sorted((run_dir / "graphql").glob("batch_*.json")):
    for node in ((read_json(p).get("data") or {}).get("repository") or {}).values():
        if node:
            fix_links[node["number"]] = [
                {"pr": r["number"], "repo": r["repository"]["nameWithOwner"], "merged": bool(r["merged"])}
                for r in node["closedByPullRequestsReferences"]["nodes"] if r]
            link_source[node["number"]] = "graphql"

print(f"Fix-link information for {len(fix_links):,} closed issues; "
      f"{sum(1 for v in fix_links.values() if v):,} have at least one linked PR")

Fix-link information for 12,205 closed issues; 4,383 have at least one linked PR


### 7e. Duplicate pairs
Issue bodies and comments are searched for *duplicate of #N*, *duplicate to #N*, *duplicates #N*, *dup/dupe of #N*,
*same as #N* and *closing in favor of #N* (full issue links count as well as `#N`).
A pair is kept only if both sides are issues in the data. A-B and B-A count as one pair, and the earliest match is kept.

In [15]:
DUP_RE = re.compile(
    r"(?P<phrase>\b(?:duplicate[sd]?\s+(?:of|to)|dup(?:e|licate)?\s+of|duplicates|same\s+as"
    r"|clos(?:e|ed|ing)\s+in\s+favou?r\s+of))\s*:?\s*(?:issue\s+)?"
    rf"(?:#|{ISSUE_URL})(?P<num>\d+)\b",
    re.IGNORECASE)

texts = [(i["number"], "issue_body", i["created_at"], anon_text(i["body"])) for i in issues_raw]
texts += [(n, "comment", c["created_at"], c["body"]) for n, cs in comments_by_issue.items() for c in cs]

pair_rows, seen = [], set()
for number, source, when, body in sorted(texts, key=lambda t: t[2]):
    for m in DUP_RE.finditer(body or ""):
        other = int(m.group("num"))
        if other == number or other not in issue_numbers or frozenset((number, other)) in seen:
            continue
        seen.add(frozenset((number, other)))
        pair_rows.append({
            "issue": number, "other_issue": other, "matched_text": m.group(0),
            "phrase": m.group("phrase").lower(), "found_in": source, "found_at": when,
            "context": " ".join(body[max(0, m.start() - 80):m.end() + 80].split()),
            "issue_url": f"https://github.com/{REPO_FULL}/issues/{number}",
            "other_issue_url": f"https://github.com/{REPO_FULL}/issues/{other}",
        })
pairs = pd.DataFrame(pair_rows, columns=["issue", "other_issue", "matched_text", "phrase", "found_in", "found_at",
                                         "context", "issue_url", "other_issue_url"])
print(f"{len(pairs):,} duplicate pairs")
pairs.head()

473 duplicate pairs


,issue,other_issue,matched_text,phrase,found_in,found_at,context,issue_url,other_issue_url
0,137,105,Duplicate of #105,duplicate of,comment,2020-03-26T23:08:47Z,Duplicate of #105,https://github.com/project-chip/connectedhomei...,https://github.com/project-chip/connectedhomei...
1,114,143,dup of #143,dup of,comment,2020-03-28T01:24:47Z,possible dup of #143?,https://github.com/project-chip/connectedhomei...,https://github.com/project-chip/connectedhomei...
2,144,143,duplicate of #143,duplicate of,comment,2020-03-28T18:04:07Z,duplicate of #143?,https://github.com/project-chip/connectedhomei...,https://github.com/project-chip/connectedhomei...
3,174,36,Duplicate of: https://github.com/project-chip/...,duplicate of,comment,2020-03-31T20:29:23Z,Duplicate of: https://github.com/project-chip/...,https://github.com/project-chip/connectedhomei...,https://github.com/project-chip/connectedhomei...
4,636,464,dup of #464,dup of,comment,2020-05-12T23:02:07Z,dup of #464,https://github.com/project-chip/connectedhomei...,https://github.com/project-chip/connectedhomei...


### 7f. One row per issue

`outcome` is assigned in this order:
1. **unresolved**: the issue is open
2. **duplicate**: closed with state_reason `duplicate`
3. **rejected**: closed with state_reason `not_planned`
4. **success**: closed and linked to a merged PR
5. **unclear**: closed without a linked merged fix

In [16]:
def pr_label(link):
    return f"#{link['pr']}" if link["repo"] == REPO_FULL else f"{link['repo']}#{link['pr']}"


def outcome(state, reason, merged):
    if state == "open":
        return "unresolved"
    if reason == "duplicate":
        return "duplicate"
    if reason == "not_planned":
        return "rejected"
    return "success" if merged else "unclear"


dup_partners, dup_text = defaultdict(list), defaultdict(list)
for r in pair_rows:
    dup_partners[r["issue"]].append(r["other_issue"])
    dup_partners[r["other_issue"]].append(r["issue"])
    dup_text[r["issue"]].append(r["matched_text"])

rows = []
for i in issues_raw:
    n = i["number"]
    cs = comments_by_issue.get(n, [])
    fix = fix_links.get(n, [])
    merged = any(l["merged"] for l in fix)
    created = pd.Timestamp(i["created_at"])
    closed = pd.Timestamp(i["closed_at"]) if i["closed_at"] else None
    rows.append({
        "number": n,
        "url": i["html_url"],
        "title": i["title"],
        "body": anon_text(i["body"]),
        "comments": "\n\n---\n\n".join(
            f"[{c['created_at']}] {c['author']}{' (bot)' if c['bot'] else ''}: {c['body']}" for c in cs),
        "labels": "; ".join(l["name"] for l in i["labels"]),
        "state": i["state"],
        "state_reason": i.get("state_reason"),
        "created_at": i["created_at"],
        "closed_at": i["closed_at"],
        "days_to_close": round((closed - created).total_seconds() / 86400, 2) if closed is not None else None,
        "comment_count": len(cs),
        "bot_comment_count": sum(c["bot"] for c in cs),
        "author_id": anon_user(i["user"]),
        "fix_pr_numbers": "; ".join(pr_label(l) for l in fix),
        "fix_merged": "yes" if merged else "no",
        "link_source": link_source.get(n, "not_checked" if i["state"] == "open" else "missing"),
        "outcome": outcome(i["state"], i.get("state_reason"), merged),
        "duplicate_pair_with": "; ".join(f"#{x}" for x in sorted(set(dup_partners[n]))),
        "duplicate_matched_text": " | ".join(dup_text[n]),
    })
df = pd.DataFrame(rows)
print(f"{len(df):,} rows x {df.shape[1]} columns")
df.head()

14,698 rows x 20 columns


,number,url,title,body,comments,labels,state,state_reason,created_at,closed_at,days_to_close,comment_count,bot_comment_count,author_id,fix_pr_numbers,fix_merged,link_source,outcome,duplicate_pair_with,duplicate_matched_text
0,2,https://github.com/project-chip/connectedhomei...,Run clang-tidy locally,How do / can I run clang-tidy locally on my Ma...,[2020-03-03T18:51:31Z] user_00008: From #1:\r\...,enhancement,closed,completed,2020-03-03T18:47:47Z,2020-03-13T19:52:36Z,10.05,7,0,user_00001,#47,yes,graphql,success,,
1,33,https://github.com/project-chip/connectedhomei...,Build warning,Warning while compiling main build: <https://g...,,bug,closed,completed,2020-03-11T04:19:00Z,2020-03-12T19:18:41Z,1.62,0,0,user_00001,#40,yes,graphql,success,,
2,34,https://github.com/project-chip/connectedhomei...,Enable travis-ci,Let's move to travis for build steps. Waiting ...,,enhancement,closed,completed,2020-03-11T04:21:17Z,2020-04-16T02:33:09Z,35.92,0,0,user_00001,#371,yes,graphql,success,,
3,35,https://github.com/project-chip/connectedhomei...,Enable restyled.io,"Config is in, we should have this enabled for ...",[2020-04-15T20:42:37Z] user_00001: Done,enhancement,closed,completed,2020-03-11T04:22:03Z,2020-04-15T20:42:37Z,35.68,1,0,user_00001,,no,graphql,unclear,,
4,36,https://github.com/project-chip/connectedhomei...,Enable codecov,"Once Marketplace access is available, we shoul...",,enhancement,closed,completed,2020-03-11T04:23:33Z,2020-04-18T14:40:40Z,38.43,0,0,user_00001,#414,yes,graphql,success,#174,


## 8. Save everything into ONE CSV file
`data/processed/matter_issues_full.csv` holds every issue with its full text and all its comments.
The CSV is UTF-8 with a BOM, so Excel shows special characters correctly.
Note: Excel shows at most 32,767 characters per cell, so very long comment threads look cut off **in Excel only**.
The file itself is complete. Load it with pandas to see everything.

In [17]:
PROCESSED.mkdir(parents=True, exist_ok=True)
df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
print("Saved", OUTPUT_CSV, f"({OUTPUT_CSV.stat().st_size / 1e6:.1f} MB)")

# Check: reading the file back gives exactly the same data.
check = pd.read_csv(OUTPUT_CSV, encoding="utf-8-sig", keep_default_na=False)
assert len(check) == len(df) and list(check.columns) == list(df.columns)
assert (check["body"] == df["body"].fillna("")).all() and (check["comments"] == df["comments"]).all()
print("Read-back check passed: all rows and all text are intact.")

Saved data/processed/matter_issues_full.csv (42.7 MB)
Read-back check passed: all rows and all text are intact.


## 9. Quick look at the data

In [18]:
summary = pd.Series({
    "Total issues": len(df),
    "Open": int((df.state == "open").sum()),
    "Closed": int((df.state == "closed").sum()),
    "Success (closed + merged fix PR)": int((df.outcome == "success").sum()),
    "Rejected (not planned)": int((df.outcome == "rejected").sum()),
    "Closed as duplicate": int((df.outcome == "duplicate").sum()),
    "Unclear (closed, no merged fix found)": int((df.outcome == "unclear").sum()),
    "Duplicate pairs": len(pairs),
    "Comments on issues": n_issue_comments,
    "Comments in whole repo (issues + PRs)": len(comments_raw),
    "Pull requests": len(prs_raw),
}, name="count")
summary.to_frame()

,count
Total issues,14698
Open,2493
Closed,12205
Success (closed + merged fix PR),4235
Rejected (not planned),40
Closed as duplicate,4
"Unclear (closed, no merged fix found)",7926
Duplicate pairs,473
Comments on issues,30379
Comments in whole repo (issues + PRs),142945


## 10. PDF overview for the professor
Saved as `data/processed/overview.pdf`.

In [19]:
BAR, INK, MUTED, GRID, PAGE = "#2a78d6", "#0b0b0b", "#52514e", "#e4e3df", (8.5, 11)
plt.rcParams.update({
    "font.size": 10, "axes.edgecolor": GRID, "axes.labelcolor": MUTED, "xtick.color": MUTED,
    "ytick.color": MUTED, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6, "axes.axisbelow": True,
    "axes.titlesize": 12, "axes.titleweight": "bold", "axes.titlecolor": INK, "axes.titlelocation": "left",
})


def text_page(pdf, title, blocks):
    """blocks: list of (text, style); style is 'p' (paragraph), 'h' (heading) or 'mono' (raw JSON)."""
    fig = plt.figure(figsize=PAGE)
    fig.text(0.07, 0.95, title, fontsize=16, weight="bold", color=INK, va="top")
    y = 0.90
    for text, style in blocks:
        if style == "mono":
            lines, size, step = text.splitlines(), 7, 0.0115
        else:
            lines = textwrap.wrap(text, 95 if style == "p" else 80) or [""]
            size, step = (11, 0.019) if style == "h" else (9.5, 0.0165)
        for line in lines:
            fig.text(0.07, y, line, fontsize=size, color=MUTED if style == "mono" else INK,
                     family="monospace" if style == "mono" else None,
                     weight="bold" if style == "h" else "normal", va="top")
            y -= step
        y -= 0.012
    pdf.savefig(fig)
    plt.close(fig)


def bar_labels(ax, bars, horizontal=False):
    for b in bars:
        v = b.get_width() if horizontal else b.get_height()
        if horizontal:
            ax.text(v, b.get_y() + b.get_height() / 2, f" {v:,.0f}", va="center", fontsize=8, color=MUTED)
        else:
            ax.text(b.get_x() + b.get_width() / 2, v, f"{v:,.0f}", ha="center", va="bottom", fontsize=8, color=MUTED)


def raw_example(n):
    """The raw API record for issue n (selected fields; login and @mentions anonymised; body shortened)."""
    i = raw_by_number[n]
    body = anon_text(i["body"]).replace("\r", "")
    keep_labels = [{"name": l["name"]} for l in i["labels"]]
    keep = {
        "number": i["number"], "html_url": i["html_url"], "title": i["title"],
        "user": {"login": anon_user(i["user"])}, "labels": "LABELS",
        "state": i["state"], "state_reason": i.get("state_reason"), "created_at": i["created_at"],
        "closed_at": i["closed_at"], "comments": i["comments"],
        "body": body[:300] + (" ..." if len(body) > 300 else ""),
    }
    text = json.dumps(keep, indent=2, ensure_ascii=False)
    text = text.replace('"labels": "LABELS"', '"labels": ' + json.dumps(keep_labels, ensure_ascii=False))   # one line
    out = []
    for line in text.splitlines():
        out.extend(textwrap.wrap(line, 110, subsequent_indent="      ") or [""])
    return "\n".join(out[:20])


raw_by_number = {i["number"]: i for i in issues_raw}
created = pd.to_datetime(df["created_at"])
outcomes = df["outcome"].value_counts()
median_days = df["days_to_close"].median()
labels = df["labels"].fillna("").str.split("; ").explode()
labels = labels[labels != ""]

with PdfPages(PROCESSED / "overview.pdf") as pdf:
    # Page 1: summary table
    fig = plt.figure(figsize=PAGE)
    fig.text(0.07, 0.95, "Matter (connectedhomeip) GitHub issues - data overview", fontsize=16, weight="bold", color=INK, va="top")
    intro = (f"Source: github.com/project-chip/connectedhomeip, downloaded on {download_date} with the GitHub REST and "
             "GraphQL APIs. One row per issue; pull requests are counted separately. Usernames are replaced by "
             "anonymous IDs. An issue counts as 'success' when GitHub links it to a merged pull request that closed it.")
    for k, line in enumerate(textwrap.wrap(intro, 95)):
        fig.text(0.07, 0.90 - k * 0.018, line, fontsize=10, color=INK, va="top")
    ax = fig.add_axes([0.07, 0.42, 0.86, 0.36])
    ax.axis("off")
    tbl = ax.table(cellText=[[k, f"{v:,}"] for k, v in summary.items()], colLabels=["Measure", "Count"],
                   colWidths=[0.75, 0.25], loc="upper left", cellLoc="left")
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(10)
    tbl.scale(1, 1.6)
    for (r, c), cell in tbl.get_celld().items():
        cell.set_edgecolor(GRID)
        if r == 0:
            cell.set_text_props(weight="bold")
    notes = [f"Median time to close: {median_days:.1f} days (closed issues only).",
             f"Issues created from {created.min():%B %Y} to {created.max():%B %Y}.",
             "'Unclear' does not mean unfixed: many issues are fixed by PRs that do not use a",
             "closing keyword, so GitHub has no formal link between them."]
    for k, line in enumerate(notes):
        fig.text(0.07, 0.36 - k * 0.02, line, fontsize=10, color=INK, va="top")
    pdf.savefig(fig)
    plt.close(fig)

    # Page 2: issues per year + outcomes
    fig, (a1, a2) = plt.subplots(2, 1, figsize=PAGE)
    fig.subplots_adjust(left=0.12, right=0.92, top=0.93, bottom=0.07, hspace=0.35)
    per_year = df.groupby(created.dt.year).size()
    bar_labels(a1, a1.bar(per_year.index.astype(str), per_year.values, color=BAR, width=0.6))
    a1.set_title("Issues opened per year")
    a1.set_ylabel("Issues")
    a1.grid(axis="x", visible=False)
    order = [o for o in ["success", "unclear", "duplicate", "rejected", "unresolved"] if o in outcomes]
    bar_labels(a2, a2.bar(order, [outcomes[o] for o in order], color=BAR, width=0.6))
    a2.set_title("Issue outcomes")
    a2.set_ylabel("Issues")
    a2.grid(axis="x", visible=False)
    pdf.savefig(fig)
    plt.close(fig)

    # Page 3: labels + time to close
    fig, (a1, a2) = plt.subplots(2, 1, figsize=PAGE)
    fig.subplots_adjust(left=0.28, right=0.92, top=0.93, bottom=0.07, hspace=0.3)
    top = labels.value_counts().head(15)[::-1]
    bar_labels(a1, a1.barh(top.index, top.values, color=BAR, height=0.6), horizontal=True)
    a1.set_title("15 most common labels")
    a1.set_xlabel("Issues with this label")
    a1.grid(axis="y", visible=False)
    a2.hist(df["days_to_close"].dropna().clip(upper=730), bins=60, color=BAR, edgecolor="white", linewidth=0.5)
    a2.axvline(median_days, color=INK, linewidth=1.5, linestyle="--")
    a2.text(median_days, a2.get_ylim()[1] * 0.95, f"  median {median_days:.1f} days", color=INK, fontsize=9, va="top")
    a2.set_title("Time to close (closed issues)")
    a2.set_xlabel("Days from opening to closing (issues over 730 days are counted in the last bar)")
    a2.set_ylabel("Issues")
    pdf.savefig(fig)
    plt.close(fig)

    # Page 4: three raw examples
    by_n = df.set_index("number")
    examples = []
    for oc in ["success", "duplicate", "unresolved"]:
        cand = by_n[(by_n.outcome == oc) & by_n.body.fillna("").str.len().between(200, 3000)]
        if len(cand):
            examples.append(cand.index[len(cand) // 2])
    blocks = [("Three issues as returned by the GitHub API (a subset of the ~30 fields per issue; body cut to "
               "300 characters; usernames replaced by anonymous IDs).", "p")]
    for n in examples:
        blocks += [(f"Issue #{n} - outcome: {by_n.loc[n, 'outcome']}", "h"), (raw_example(n), "mono")]
    text_page(pdf, "What the raw data looks like", blocks)

    # Page 5: duplicate pairs + messy data
    blocks = [("Pairs were found by searching issue bodies and comments for phrases such as 'duplicate of #N', "
               "'dup of #N', 'same as #N' and 'closing in favor of #N'. Both sides must be issues in the data.", "p")]
    for r in pairs[pairs.phrase.str.startswith("duplicate")].head(3).itertuples():
        blocks += [(f"#{r.issue}  <->  #{r.other_issue}   (matched: \"{r.matched_text}\")", "h"),
                   (f"#{r.issue}: {by_n.loc[r.issue, 'title']}", "p"),
                   (f"#{r.other_issue}: {by_n.loc[r.other_issue, 'title']}", "p"),
                   (r.issue_url, "p"), (r.other_issue_url, "p")]
    body = df["body"].fillna("")
    all_c = [c for cs in comments_by_issue.values() for c in cs]
    bot_share = sum(c["bot"] for c in all_c) / max(1, len(all_c))
    blocks += [
        ("What is messy in the data", "h"),
        (f"Templates: {body.str.count(r'(?m)^#{2,4} ').ge(2).mean():.0%} of issue bodies follow a form with several "
         "headings (e.g. 'Reproduction steps', 'Platform'), so many bodies share the same boilerplate text.", "p"),
        (f"Logs and code: {body.str.contains(r'```|CHIP:[A-Z]{2,4}:', regex=True).mean():.0%} of bodies contain code "
         f"blocks or device logs; {(body.str.len() > 10000).mean():.1%} are longer than 10,000 characters.", "p"),
        (f"Bot comments: {bot_share:.1%} of issue comments come from bot accounts (e.g. stale-issue bots, CI). "
         "They are marked '(bot)' in the comments column.", "p"),
        (f"Uneven labels: {(df['labels'].fillna('') == '').mean():.0%} of issues have no label; there are "
         f"{labels.nunique()} distinct labels and the top 5 make up "
         f"{labels.value_counts().head(5).sum() / max(1, len(labels)):.0%} of all label uses.", "p"),
        ("Links to fixes: 'success' only sees PRs that GitHub formally links (closing keywords or manual "
         "linking). Many real fixes are not linked, which makes 'unclear' larger.", "p"),
        ("Duplicate phrases: 'same as #N' sometimes means 'similar to', so a few pairs are not true duplicates.", "p"),
        (f"Close reasons: few issues use GitHub's 'closed as duplicate' ({(df.outcome == 'duplicate').sum()}) or "
         f"'not planned' ({(df.outcome == 'rejected').sum()}) reasons. Instead, "
         f"{((df.state == 'closed') & df['labels'].str.contains(r'(?:^|; )stale(?:;|$)')).sum():,} closed issues carry "
         "the 'stale' label (closed for inactivity); these count as 'unclear'.", "p"),
    ]
    text_page(pdf, "Duplicate pairs and data quality", blocks)

print("Saved", PROCESSED / "overview.pdf")

Saved data/processed/overview.pdf


## 11. Final counts and file locations

In [20]:
print(summary.to_string())
print()
print("One-file dataset :", OUTPUT_CSV.resolve())
print("PDF overview     :", (PROCESSED / "overview.pdf").resolve())
print("Raw API responses:", run_dir.resolve())

Total issues                              14698
Open                                       2493
Closed                                    12205
Success (closed + merged fix PR)           4235
Rejected (not planned)                       40
Closed as duplicate                           4
Unclear (closed, no merged fix found)      7926
Duplicate pairs                             473
Comments on issues                        30379
Comments in whole repo (issues + PRs)    142945
Pull requests                             32841

One-file dataset : /Users/tejas/Downloads/Crazy-Stuff/Finding-Old-Fixes-For-New-Failures/data/processed/matter_issues_full.csv
PDF overview     : /Users/tejas/Downloads/Crazy-Stuff/Finding-Old-Fixes-For-New-Failures/data/processed/overview.pdf
Raw API responses: /Users/tejas/Downloads/Crazy-Stuff/Finding-Old-Fixes-For-New-Failures/data/raw/download_2026-11-11
